# Week 6 EXERCISE: Game Publisher Submission Triage

**Goal:** Help a game publisher prioritize incoming pitches — classify each as **High**, **Medium**, or **Low** priority.

**Flow:**
1. **Dataset** — Built-in (pitch, priority) examples; optionally generate more with an LLM via **OpenRouter**.
2. **Frontier baseline** — Run zero-shot **OpenRouter** on the test set and report **accuracy** (the number to beat).
3. **Export JSONL** — Save training data to `jsonl/` for use elsewhere (e.g. open-source fine-tuning) if you like.
4. **Evaluate** — Report baseline accuracy and show sample predictions.


In [57]:
# imports

import os
import json
import random
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
FRONTIER_MODEL = "openai/gpt-4o-mini"

api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)

## Dataset: game submission pitches and priority

Synthetic (pitch, priority) examples. **High** = strong fit, clear hook, near-complete. **Medium** = solid but generic or early. **Low** = very early, vague, or mismatched.

In [58]:
PRIORITIES = ("High", "Medium", "Low")

SUBMISSIONS = [
    {"pitch": "Roguelike deckbuilder, 80% complete, Steam page live. Strong wishlist. Team of 3, shipped 2 games. Looking for QA and marketing support for Q2 launch.", "priority": "High"},
    {"pitch": "Narrative puzzle game, vertical slice done, targeting Switch and PC. Art style similar to Gris. Need funding for 8 months to release.", "priority": "High"},
    {"pitch": "Co-op roguelike for 2–4 players, alpha in 3 months. Clear scope, proven genre. Seeking advance and porting help for console.", "priority": "High"},
    {"pitch": "Pixel art metroidvania, demo on Steam. 50k wishlists. Solo dev, 12 months to launch. Need QA and localization.", "priority": "High"},
    {"pitch": "Tactics RPG with job system, beta in 2 months. Full design doc and GDD. Team of 5, one shipped title. PC first, then Switch.", "priority": "High"},
    {"pitch": "Idle clicker with narrative twist, soft launch data positive. Mobile + Steam. Looking for publisher for global release.", "priority": "High"},
    {"pitch": "Horror walking sim, vertical slice and trailer ready. Strong mood and story. 6 months to beta. PC and console.", "priority": "High"},
    {"pitch": "Local multiplayer party game, 4 players, alpha playable. Ideal for Switch. Team of 2, previous jam experience.", "priority": "High"},
    {"pitch": "Turn-based strategy with hex map, 70% complete. Similar to Into the Breach. Need marketing and store presence.", "priority": "High"},
    {"pitch": "Narrative adventure, branching dialogue, demo on itch. Strong writing sample. 9 months to release. PC only.", "priority": "High"},
    {"pitch": "We are making a cool RPG game. It will have lots of quests and fighting. Still in early development. No demo yet.", "priority": "Low"},
    {"pitch": "My friend and I have an idea for a multiplayer shooter. We have some concept art. Looking for funding to start development.", "priority": "Low"},
    {"pitch": "Open-world survival game with crafting. Very early stage. We think it could be like Minecraft meets Rust. No playable build.", "priority": "Low"},
    {"pitch": "Mobile puzzle game. We have a prototype but no clear hook. Would like publisher support for everything.", "priority": "Low"},
    {"pitch": "An MMO with unique combat. Still in pre-production. Need full funding and team. No vertical slice.", "priority": "Low"},
    {"pitch": "We want to make the next big battle royale. Concept only. No technical demo. Looking for full budget.", "priority": "Low"},
    {"pitch": "Casual mobile game, idea stage. No build. Need publisher to help with design and development.", "priority": "Low"},
    {"pitch": "RPG with 100 hours of content. Solo dev, 6 months in, no playable demo. Would need long timeline.", "priority": "Low"},
    {"pitch": "Social deduction game for 10+ players. Early prototype, unproven on market. No clear platform.", "priority": "Low"},
    {"pitch": "Retro platformer, very early. We have a short level. Looking for advance to work full-time for 2 years.", "priority": "Low"},
    {"pitch": "2D platformer with metroidvania elements. Demo available, decent controls. Genre is crowded. No clear differentiator yet.", "priority": "Medium"},
    {"pitch": "Puzzle game inspired by Portal. Prototype works but art is placeholder. Need 6 months and funding to polish.", "priority": "Medium"},
    {"pitch": "Tower defense with RPG progression. Alpha in progress. Solid mechanics, art style not final. PC first.", "priority": "Medium"},
    {"pitch": "Narrative game with choices. Writing is strong, gameplay is light. Vertical slice in 4 months. Small scope.", "priority": "Medium"},
    {"pitch": "Roguelike shooter, early alpha. Fun core loop, needs content and balance. Team of 2, first commercial game.", "priority": "Medium"},
    {"pitch": "Co-op puzzle adventure. Concept is clear, build is 30% done. Would need 12 months and QA support.", "priority": "Medium"},
    {"pitch": "Deckbuilder with unique theme. Prototype playable, balance rough. Looking for feedback and possible deal.", "priority": "Medium"},
    {"pitch": "Horror puzzle game. Mood is good, puzzles need work. No release date. Solo dev, part-time.", "priority": "Medium"},
    {"pitch": "Action RPG, alpha stage. Combat feels good, story and scope still growing. Need 18 months.", "priority": "Medium"},
    {"pitch": "Party game for 2–4 players. Mini-games are fun, presentation is early. Targeting Switch, 9 months to beta.", "priority": "Medium"},
    {"pitch": "Souls-like 2D, hard difficulty. Niche but dedicated. Demo on Steam, modest wishlist. Need marketing.", "priority": "High"},
    {"pitch": "Management sim, city-builder hybrid. Beta in 4 months. Data from beta testers positive. PC + console.", "priority": "High"},
    {"pitch": "Story-driven RPG, no combat. Demo and trailer ready. Strong writing, 6 months to launch. PC only.", "priority": "High"},
    {"pitch": "Rhythm game with custom tracks. Community tools ready. Early access in 2 months. Niche but engaged.", "priority": "High"},
    {"pitch": "Tactical stealth game, vertical slice done. Clear vision, small team. Need QA and porting for console.", "priority": "High"},
    {"pitch": "Colony sim with survival elements. Alpha playable, roadmap clear. 12 months to 1.0. PC first.", "priority": "High"},
    {"pitch": "We have a dream to make an open-world game. No team, no demo, no design doc. Just an idea.", "priority": "Low"},
    {"pitch": "Multiplayer only game, no single player. Very early prototype. Unclear monetization.", "priority": "Low"},
    {"pitch": "Sequel to our unreleased first game. First game had no traction. Looking for publisher for both.", "priority": "Low"},
    {"pitch": "Fighting game with 20 characters. Two people, 1 year in. No animator. Scope is very large.", "priority": "Low"},
    {"pitch": "Educational game for kids. Noble goal but no gameplay hook. Early prototype.", "priority": "Low"},
    {"pitch": "Top-down shooter, functional prototype. Generic theme. Could be polished in 6 months with support.", "priority": "Medium"},
    {"pitch": "Visual novel with mini-games. Script 50% done. Art style consistent. Need 8 months and voice budget.", "priority": "Medium"},
    {"pitch": "Racing game with power-ups. Core mechanics work, content thin. First project for team of 3.", "priority": "Medium"},
    {"pitch": "Dungeon crawler, tile-based. Alpha available. Classic feel, needs UX pass and content.", "priority": "Medium"},
    {"pitch": "Endless runner with roguelike elements. Mobile + PC. Prototype is fun, monetization not decided.", "priority": "Medium"},
    {"pitch": "Automation game, factorio-like. Complex systems, alpha. Long development ahead, small team.", "priority": "Medium"},
    {"pitch": "Bullet hell shooter. Tight controls, content in progress. Niche audience. 6 months to early access.", "priority": "Medium"},
    {"pitch": "Mystery adventure, point-and-click. Story outlined, first chapter playable. Need funding for full game.", "priority": "Medium"},
    {"pitch": "Twin-stick shooter with upgrades. Alpha, 4 months to beta. Indie scope, clear mechanics.", "priority": "Medium"},
    {"pitch": "Card battler with narrative. Demo on itch. Unique theme. Need 10 months and localization.", "priority": "High"},
    {"pitch": "Platform fighter, local + online. Netcode in progress. Beta in 5 months. Dedicated genre audience.", "priority": "High"},
    {"pitch": "Farming sim with RPG. Co-op planned. Vertical slice in 2 months. Strong reference (Stardew).", "priority": "High"},
    {"pitch": "Narrative puzzle horror. Demo and trailer ready. Small scope, 6 months. PC + console.", "priority": "High"},
    {"pitch": "Tactics game with permadeath. Alpha, clear design. 9 months to release. Need QA and marketing.", "priority": "High"},
    {"pitch": "Roguelike with daily runs. Meta-progression designed. Beta in 3 months. Mobile + PC potential.", "priority": "High"},
    {"pitch": "Puzzle platformer, physics-based. Demo polished. 4 months to launch. Solo dev, one previous release.", "priority": "High"},
    {"pitch": "Co-op horror, 2 players. Vertical slice done. Strong atmosphere. Need 8 months and porting.", "priority": "High"},
    {"pitch": "Strategy game, turn-based, hex. Beta in 1 month. Mod support planned. PC focus.", "priority": "High"},
    {"pitch": "Action platformer with speedrun focus. Demo on Steam. Community interest. 5 months to 1.0.", "priority": "High"},
    {"pitch": "Idle/incremental with narrative. Soft launch data good. Small scope. Need publisher for store and marketing.", "priority": "High"},
    {"pitch": "A game where you do stuff. We have ideas. Maybe 2 years of development. No prototype.", "priority": "Low"},
    {"pitch": "Battle royale with a twist. Pre-alpha. Large scope. Need full funding and team expansion.", "priority": "Low"},
    {"pitch": "VR game, concept only. No headset build. Looking for VR publisher and budget.", "priority": "Low"},
    {"pitch": "Sequel to a game that didn't sell. We believe in the IP. Need advance to finish.", "priority": "Low"},
    {"pitch": "Multiplayer only, 50 players per match. Small team. No technical demo. Very ambitious.", "priority": "Low"},
    {"pitch": "RPG with procedural world. Huge scope. One programmer. No vertical slice after 1 year.", "priority": "Low"},
    {"pitch": "Sports game, no license. Early prototype. Niche. Unclear differentiator.", "priority": "Medium"},
    {"pitch": "Narrative RPG, text-heavy. Good writing sample. Playable chapter 1. Need 14 months.", "priority": "Medium"},
    {"pitch": "Roguelike with crafting. Systems in place, content needed. First game, small team.", "priority": "Medium"},
    {"pitch": "Puzzle game with story. Mechanics proven, narrative in progress. 7 months to beta.", "priority": "Medium"},
    {"pitch": "Tower defense + RPG. Alpha. Fun core loop, needs variety. PC and mobile possible.", "priority": "Medium"},
    {"pitch": "Exploration game, minimal UI. Mood over mechanics. Vertical slice in 5 months. Art-heavy.", "priority": "Medium"},
    {"pitch": "Shooter with roguelike runs. Alpha. Good feel, content pipeline unclear. 10 months estimate.", "priority": "Medium"},
    {"pitch": "Social deduction, 6–8 players. Prototype tested at events. Needs content and polish.", "priority": "Medium"},
    {"pitch": "Racing + combat. Cars and weapons. Alpha. Niche genre. 8 months to early access.", "priority": "Medium"},
    {"pitch": "Base-building strategy. Single player focus. Alpha in 2 months. Clear scope, small team.", "priority": "Medium"},
]

Split the dataset into **train** (70%), **validation** (15%), and **test** (15%) with a fixed seed for reproducibility.

In [ ]:
random.seed(42)
shuffled = SUBMISSIONS.copy()
random.shuffle(shuffled)
n = len(shuffled)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_data = shuffled[:n_train]
val_data = shuffled[n_train : n_train + n_val]
test_data = shuffled[n_train + n_val :]
print(f"Train {len(train_data)}, Val {len(val_data)}, Test {len(test_data)}")

Generate more (pitch, priority) examples with the LLM

In [ ]:
def generate_triage_examples(n_per_priority: int = 2):
    """Generate (pitch, priority) examples via OpenRouter. Appends to SUBMISSIONS."""
    prompt = f"""Generate {n_per_priority} short game submission pitches for each priority: High, Medium, Low.
High = strong fit, near-complete, clear hook. Medium = solid but generic or early. Low = very early, vague, or mismatched.
Reply with a JSON array of objects: [{{"pitch": "one sentence pitch", "priority": "High"}}, ...]
Only the JSON array, no other text."""
    r = client.chat.completions.create(model=FRONTIER_MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=800)
    raw = (r.choices[0].message.content or "").strip()
    raw = re.sub(r"^```(?:json)?\\s*", "", raw).strip()
    raw = re.sub(r"\\s*```$", "", raw).strip()
    try:
        extra = json.loads(raw)
        for ex in extra:
            if isinstance(ex, dict) and ex.get("priority") in PRIORITIES and ex.get("pitch"):
                SUBMISSIONS.append(ex)
        print(f"Added {len(extra)} generated examples. Total: {len(SUBMISSIONS)}")
    except Exception as e:
        print(f"Generation failed: {e}")
generate_triage_examples(2)  # uncomment to run

Run the zero-shot classifier via OpenRouter on the test set and report accuracy.

In [ ]:
PRIORITIES = ("High", "Medium", "Low")

def predict_baseline(pitch: str) -> str:
    """Zero-shot classification via OpenRouter (frontier baseline)."""
    sys_prompt = (
        "You are a game publisher triaging submissions. "
        "Classify the following pitch as exactly one of: High, Medium, Low. "
        "Reply with only that one word, nothing else."
    )
    r = client.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": pitch},
        ],
        max_tokens=10,
    )
    raw = (r.choices[0].message.content or "").strip()
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    return raw or "Medium"

def accuracy(predictor, data):
    correct = sum(1 for ex in data if predictor(ex["pitch"]) == ex["priority"])
    return correct / len(data) if data else 0.0

baseline_acc = accuracy(predict_baseline, test_data)
print(f"=== Frontier baseline (OpenRouter zero-shot) ===")
print(f"Accuracy on test set: {baseline_acc:.1%}")
print("This is the number to beat with fine-tuning.")

Prepare training data.

In [62]:
def messages_for(example):
    """One training example: user = pitch, assistant = priority label."""
    return [
        {"role": "user", "content": example["pitch"]},
        {"role": "assistant", "content": example["priority"]},
    ]


def make_jsonl(items):
    lines = [json.dumps({"messages": messages_for(ex)}) for ex in items]
    return "\n".join(lines)


def write_jsonl(items, filepath):
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))

Export to jsonl folder which must exist before hand

In [ ]:
JSONL_DIR = "jsonl"
write_jsonl(train_data, f"{JSONL_DIR}/fine_tune_train.jsonl")
write_jsonl(val_data, f"{JSONL_DIR}/fine_tune_validation.jsonl")
print(f"Wrote {len(train_data)} train, {len(val_data)} val to {JSONL_DIR}/")

The files in `jsonl/` can be used with OpenAI fine-tuning (if you have a key) or with open-source tools. This notebook uses OpenRouter only.

In [ ]:
# JSONL is already in jsonl/. Use it with OpenAI (platform.openai.com) or open-source fine-tuning if you like.
print(f"Training data: {JSONL_DIR}/fine_tune_train.jsonl ({len(train_data)} examples), {JSONL_DIR}/fine_tune_validation.jsonl ({len(val_data)} examples)")

Evaluate (OpenRouter baseline accuracy)

In [ ]:
def accuracy(predictor, data):
    correct = sum(1 for ex in data if predictor(ex["pitch"]) == ex["priority"])
    return correct / len(data) if data else 0.0


acc_baseline = accuracy(predict_baseline, test_data)
print(f"Frontier baseline (OpenRouter): {acc_baseline:.1%}")

Reults which show the best pitch deck submissions that a game publisher can use to select who gets funding.

In [ ]:
# Show a few test predictions (OpenRouter baseline)
for ex in test_data[:5]:
    pred = predict_baseline(ex["pitch"])
    print(f"True: {ex['priority']} | Predicted: {pred}")
    print(f"  {ex['pitch'][:80]}...")
    print()